# KEIBA データ取得ノートブック

このノートブックは、開催日・レース・馬・血統データを順に取得して生データテーブルを更新するための実行手順をまとめたものです。

ステップは以下の順序で進みます。

1. 開催日一覧の取得（2.1）
2. レースIDの取得およびレースHTMLの保存・生テーブルへの反映（2.2）
3. 馬IDの取得および馬HTMLの保存・生テーブルへの反映（2.3）
4. 血統HTMLの保存・生テーブルへの反映（2.4）

日付範囲を初回から取得する場合は、`FROM_DATE` と `TO_DATE` を設定して実行してください。既に一部データを取得済みの場合は、既存CSVやHTMLを再利用するため、中間成果物がなくてもこのノートブック単体で一貫して動作します。

In [ ]:
# 必要なモジュールのインポート
import os
import glob
import pandas as pd
from modules import preparing
from modules.constants import LocalPaths

# ヘルパ関数の定義
def unique_string_ids(values):
    """
    リストやシリーズ中のNoneや空文字を除外し、重複を除いた文字列リストを返します。
    """
    if values is None:
        return []
    s = pd.Series(list(values), dtype="object").dropna().astype(str).str.strip()
    s = s[s != ""]
    if s.empty:
        return []
    return s.drop_duplicates().tolist()

def collect_existing_html_files(base_dir, entity_ids):
    files = []
    for entity_id in unique_string_ids(entity_ids):
        matched = sorted(glob.glob(os.path.join(base_dir, f'{entity_id}*.bin')))
        if matched:
            files.append(matched[0])
    return files

def update_rawdata_if_any(filepath, new_df, label):
    """
    取得結果のデータフレームが存在する場合に raw データファイルを更新します。
    """
    if new_df is None:
        print(f'{label}: skipped (new_df is None)')
        return
    if hasattr(new_df, 'empty') and new_df.empty:
        print(f'{label}: skipped (0 rows)')
        return
    preparing.update_rawdata(filepath, new_df)
    print(f'{label}: updated ({len(new_df)} rows) -> {filepath}')

In [ ]:
# 取得する期間を設定してください。
# 例: '2010/01/01' から '2026/04/12'
FROM_DATE = '2010/01/01'
TO_DATE = '2026/04/12'
# Selenium 待ち時間やカレンダーアクセス間隔などのパラメータ
WAITING_TIME = 10  # レースID取得時の待ち秒数
CALENDAR_SLEEP_SECONDS = 1.0  # カレンダー画面アクセス間隔(秒)
DOWNLOAD_RACE_HTML = True  # レースHTMLを保存するか
OVERWRITE_HTML = False  # 既存HTMLを上書きするか

In [ ]:
# ステップ2.1: 開催日・レースIDの取得
print('=== step 2.1: scrape kaisai dates ===')
kaisai_date_list = preparing.scrape_kaisai_date(from_=FROM_DATE, to_=TO_DATE, sleep_seconds=CALENDAR_SLEEP_SECONDS)
print(f'kaisai dates: {len(kaisai_date_list)}
', kaisai_date_list)

print('=== step 2.1: scrape race ids ===')
race_id_list = preparing.scrape_race_id_list(kaisai_date_list, waiting_time=WAITING_TIME, save_csv_path=None, continue_on_error=True, dedupe=True)
print(f'race ids: {len(race_id_list)}')

In [ ]:
# ステップ2.2: レースHTMLの取得と raw テーブル更新
print('=== step 2.2: scrape /race and update raw tables ===')
# HTML保存ディレクトリを解決
race_html_dir = getattr(LocalPaths, 'HTML_RACE_DIR', os.path.join(LocalPaths.DATA_DIR, 'html', 'race'))
# レースHTMLの保存または既存ファイルの収集
if DOWNLOAD_RACE_HTML:
    html_files_race = preparing.scrape_html_race(race_id_list, skip=not OVERWRITE_HTML)
else:
    html_files_race = collect_existing_html_files(race_html_dir, race_id_list)
html_files_race = unique_string_ids(html_files_race)
print(f'race html files count: {len(html_files_race)}')
if not html_files_race:
    raise RuntimeError('race HTML が1件も存在しません。DOWNLOAD_RACE_HTML=True にするか、htmlファイルを指定ディレクトリに配置してください。')

# レースHTMLから生データテーブルを更新
results_new = preparing.get_rawdata_results(html_files_race)
race_info_new = preparing.get_rawdata_info(html_files_race)
return_tables_new = preparing.get_rawdata_return(html_files_race)
update_rawdata_if_any(LocalPaths.RAW_RESULTS_PATH, results_new, 'raw results')
update_rawdata_if_any(LocalPaths.RAW_RACE_INFO_PATH, race_info_new, 'raw race info')
update_rawdata_if_any(LocalPaths.RAW_RETURN_TABLES_PATH, return_tables_new, 'raw return tables')

# 次のステップで必要なhorse_idの抽出
horse_id_list = []
if isinstance(results_new, pd.DataFrame) and 'horse_id' in results_new.columns:
    horse_id_list = unique_string_ids(results_new['horse_id'].tolist())
print(f'horse ids extracted: {len(horse_id_list)}')

In [ ]:
# ステップ2.3: 馬HTMLの取得と raw テーブル更新
print('=== step 2.3: scrape /horse and update raw tables ===')
if not horse_id_list:
    print('horse ids not found. skip horse/ped pipeline.')
else:
    horse_html_dir = getattr(LocalPaths, 'HTML_HORSE_DIR', os.path.join(LocalPaths.DATA_DIR, 'html', 'horse'))
    # 馬HTMLの取得または既存ファイルの収集
    html_files_horse = preparing.scrape_html_horse_with_master(horse_id_list, skip=not OVERWRITE_HTML)
    if not html_files_horse:
        html_files_horse = collect_existing_html_files(horse_html_dir, horse_id_list)
    html_files_horse = unique_string_ids(html_files_horse)
    print(f'horse html files count: {len(html_files_horse)}')
    if html_files_horse:
        horse_info_new = preparing.get_rawdata_horse_info(html_files_horse)
        horse_results_new = preparing.get_rawdata_horse_results(html_files_horse)
        update_rawdata_if_any(LocalPaths.RAW_HORSE_INFO_PATH, horse_info_new, 'raw horse info')
        update_rawdata_if_any(LocalPaths.RAW_HORSE_RESULTS_PATH, horse_results_new, 'raw horse results')
    else:
        print('horse HTML が見つからないため生データ更新をスキップします。')

In [ ]:
# ステップ2.4: 血統HTMLの取得と raw テーブル更新
print('=== step 2.4: scrape /ped and update raw tables ===')
if not horse_id_list:
    print('horse ids not found. skip ped pipeline.')
else:
    ped_html_dir = getattr(LocalPaths, 'HTML_PED_DIR', None) or getattr(LocalPaths, 'HTML_PEDS_DIR', None) or os.path.join(LocalPaths.DATA_DIR, 'html', 'ped')
    html_files_ped = preparing.scrape_html_ped(horse_id_list, skip=not OVERWRITE_HTML)
    if not html_files_ped:
        html_files_ped = collect_existing_html_files(ped_html_dir, horse_id_list)
    html_files_ped = unique_string_ids(html_files_ped)
    print(f'ped html files count: {len(html_files_ped)}')
    if html_files_ped:
        peds_new = preparing.get_rawdata_peds(html_files_ped)
        update_rawdata_if_any(LocalPaths.RAW_PEDS_PATH, peds_new, 'raw peds')
    else:
        print('ped HTML が見つからないため生データ更新をスキップします。')